In [ ]:
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sentence_transformers import SentenceTransformer

In [33]:
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\floco\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\floco\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\floco\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [34]:
df = pd.read_csv('top_movies.csv')

In [35]:
df

,movie_name,genre,description
0,The Shawshank Redemption,"Drama, Crime",Imprisoned in the 1940s for the double murder ...
1,The Godfather,"Drama, Crime","Spanning the years 1945 to 1955, a chronicle o..."
2,The Godfather Part II,"Drama, Crime",In the continuing saga of the Corleone crime f...
3,Schindler's List,"Drama, History, War",The true story of how businessman Oskar Schind...
4,12 Angry Men,Drama,The defense and the prosecution have rested an...
...,...,...,...
9415,The Misfits,"Action, Thriller",After being recruited by a group of unconventi...
9416,A Serbian Film,"Crime, Horror, Thriller",Retired porn star Milos leads a normal family ...
9417,The Night Before the Exams Today,Comedy,"In 2006, as World Cup fever sweeps Italy, high..."
9418,Problemos,Comedy,"When a pandemic strikes the world, Victor and ..."


In [36]:
def preprocess_text(text):
    """
    Prétraiter le texte : convertir en minuscules, tokeniser, supprimer les stop-words,
    lemmatiser, et retourner une chaîne propre.
    """
    # Convertir en minuscules
    text = text.lower()
    # Tokenisation
    tokens = word_tokenize(text)
    # Supprimer les stop-words
    stop_words = set(stopwords.words('english'))
    tokens = [word for word in tokens if word not in stop_words]
    # Lemmatisation
    lemmatizer = WordNetLemmatizer()
    tokens = [lemmatizer.lemmatize(word) for word in tokens if word.isalnum()]
    return ' '.join(tokens)


In [50]:
df['clean_description'] = df['description'].apply(preprocess_text)

In [38]:
tfidf_vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
tfidf_matrix = tfidf_vectorizer.fit_transform(df['clean_description'])

In [39]:
model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode(df['clean_description'].tolist())
cosine_sim_embeddings = cosine_similarity(embeddings)

In [40]:
cosine_sim_tfidf = cosine_similarity(tfidf_matrix)

In [41]:
def recommend_movies(movie_title, cosine_sim,num_recommendations=5):
    """
    Recommander des films similaires à un titre donné.
    Retourne une liste de films avec leur titre, genre et description.
    """
    # Vérifier si le film existe dans le dataset
    if movie_title not in df['movie_name'].values:
        return f"Le film '{movie_title}' n'est pas dans le dataset."
    
    # Trouver l'index du film
    movie_idx = df[df['movie_name'] == movie_title].index[0]
    
    # Obtenir les scores de similarité pour ce film
    sim_scores = list(enumerate(cosine_sim[movie_idx]))
    
    # Trier par score de similarité (descendant) et exclure le film lui-même
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)[1:num_recommendations+1]
    
    # Obtenir les indices des films recommandés
    movie_indices = [idx for idx, _ in sim_scores]
    
    # Retourner les films recommandés
    recommendations = df[['movie_name', 'genre', 'description']].iloc[movie_indices]
    recommendations['similarity_score'] = [score for _, score in sim_scores]
    
    return recommendations

In [42]:
movie_title = "The Godfather"  
print(f"Recommandations pour '{movie_title}':")
print(recommend_movies(movie_title, cosine_sim=cosine_sim_embeddings))

Recommandations pour 'The Godfather':
                  movie_name                               genre  \
2      The Godfather Part II                        Drama, Crime   
1624  The Godfather Part III              Crime, Drama, Thriller   
6552            The Punisher                Action, Crime, Drama   
5121     My Name Is Vendetta  Action, Crime, Adventure, Thriller   
265   Rocco and His Brothers                      Drama, Romance   

                                            description  similarity_score  
2     In the continuing saga of the Corleone crime f...          0.736840  
1624  In the midst of trying to legitimize his busin...          0.509860  
6552  When undercover FBI agent Frank Castle's wife ...          0.508410  
5121  After old enemies kill his family, a former ma...          0.493079  
265   When a impoverished widow’s family moves to th...          0.492708  


In [47]:
# Vérifier les valeurs manquantes ou non-valides dans la colonne 'genre'
print("Vérification des valeurs manquantes dans 'genre':")
print(df['genre'].isna().sum(), "valeurs manquantes")
print("Types de données dans 'genre':", df['genre'].apply(type).value_counts())

# Remplacer les NaN ou valeurs non-chaînes par une chaîne vide
df['genre'] = df['genre'].fillna('').astype(str)

Vérification des valeurs manquantes dans 'genre':
3 valeurs manquantes
Types de données dans 'genre': genre
<class 'str'>      9417
<class 'float'>       3
Name: count, dtype: int64


In [45]:
# Fonction pour calculer la similarité des genres (Jaccard)
def genre_similarity(genres1, genres2):
    """
    Calculer la similarité Jaccard entre deux ensembles de genres.
    """
    set1 = set(genres1.split(', '))
    set2 = set(genres2.split(', '))
    intersection = len(set1.intersection(set2))
    union = len(set1.union(set2))
    return intersection / union if union > 0 else 0

In [43]:
# Fonction de recommandation avec pondération des genres
def recommend_movies_2(movie_title, cosine_sim, num_recommendations=5, desc_weight=0.7, genre_weight=0.3):
    """
    Recommander des films similaires à un titre donné, en combinant la similarité des
    descriptions (embeddings) et des genres (Jaccard).
    Args:
        movie_title: Titre du film.
        num_recommendations: Nombre de films à recommander.
        desc_weight: Poids de la similarité des descriptions (0 à 1).
        genre_weight: Poids de la similarité des genres (0 à 1).
    Returns:
        DataFrame avec les films recommandés, leurs genres, descriptions, et scores.
    """
    if movie_title not in df['movie_name'].values:
        return f"Le film '{movie_title}' n'est pas dans le dataset."
    
    # Trouver l'index du film
    movie_idx = df[df['movie_name'] == movie_title].index[0]
    
    # Obtenir la similarité des descriptions
    desc_sim = cosine_sim[movie_idx]
    
    # Calculer la similarité des genres pour chaque film
    target_genres = df['genre'].iloc[movie_idx]
    genre_sim = [genre_similarity(target_genres, df['genre'].iloc[i]) for i in range(len(df))]
    
    # Combiner les similarités avec les poids
    combined_sim = desc_weight * desc_sim + genre_weight * np.array(genre_sim)
    
    # Trier par score de similarité (descendant) et exclure le film lui-même
    sim_scores = list(enumerate(combined_sim))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)[1:num_recommendations+1]
    
    # Obtenir les indices des films recommandés
    movie_indices = [idx for idx, _ in sim_scores]
    
    # Retourner les films recommandés
    recommendations = df[['movie_name', 'genre', 'description']].iloc[movie_indices]
    recommendations['similarity_score'] = [score for _, score in sim_scores]
    
    return recommendations

In [48]:
movie_title = "The Godfather"  
print(f"Recommandations pour '{movie_title}':")
print(recommend_movies_2(movie_title, cosine_sim=cosine_sim_embeddings))

Recommandations pour 'The Godfather':
                     movie_name         genre  \
2         The Godfather Part II  Drama, Crime   
16                   GoodFellas  Drama, Crime   
6615  The Many Saints of Newark  Crime, Drama   
4232               A Gang Story  Crime, Drama   
6946               The Ruthless  Crime, Drama   

                                            description  similarity_score  
2     In the continuing saga of the Corleone crime f...          0.815788  
16    The true story of Henry Hill, a half-Irish, ha...          0.639725  
6615  Young Anthony Soprano is growing up in one of ...          0.593973  
4232  After growing up in a poor gypsy camp, Edmond ...          0.591576  
6946  Milan, Italy, 1967. Santo Russo, a boy of Cala...          0.590792  
